# 00 - Reproduce the paper's TMD baseline

**Run this first.** Every improvement notebook (01-04) is judged against the paper's Table 1, and that comparison is only meaningful if *our* pipeline reproduces the paper's own TMD numbers under the same protocol. This notebook trains BERT-base and one InfoGAN, runs the paper's attacks (PWWS, TextFooler, BERT-Attack with the TextDefender constraints) on the vanilla model and on TMD, and prints the result next to Table 1.

What it caches on Google Drive (shared by notebooks 01-04): the fine-tuned model, embeddings, the InfoGAN and the baseline results.

**Known differences from the paper (they matter when you read the table):**
- Pip `textattack` instead of the authors' vendored fork. The constraints are rebuilt to match `TextDefender/utils/textattack_utils.py` (K=50 candidates, max-words-perturbed, USE angular >= 0.8408 with window 15, query budget K x words), but small behavioural differences are possible.
- The authors ran a random search over InfoGAN hyperparameters and kept the best checkpoints; here a single InfoGAN is trained with the Table 6 values. A weaker GAN than theirs is possible.
- Fine-tuning uses 3 epochs and a fixed learning rate (paper: 10 epochs, lr search).
- This code has not been run by its author. Expect to fix small API issues.

If the reproduced TMD numbers land far below Table 1, fix that first: an 'improvement' over a weak reproduction says nothing about the paper's benchmark.

In [ ]:
!pip -q install transformers datasets accelerate textattack tensorflow_hub
import os, sys, subprocess
REPO = 'https://github.com/shivanshgoyaliitk/for_shreyansh.git'
BRANCH = 'claude/magical-pasteur-ptp2k2'
DIR = '/content/for_shreyansh'
url = REPO
try:  # only needed if the repo is private: add a Colab secret named GITHUB_TOKEN
    from google.colab import userdata
    url = REPO.replace('https://', 'https://' + userdata.get('GITHUB_TOKEN') + '@')
except Exception:
    pass
if os.path.exists(DIR):
    subprocess.run(['git', '-C', DIR, 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, DIR], check=True)
sys.path.insert(0, DIR)
# If pip reports a numpy / protobuf conflict, restart the runtime once and re-run from here.

In [ ]:
import torch, tmd_lab as T
T.setup_env()
from google.colab import drive
drive.mount('/content/drive')

DATASET = 'ag_news'   # 'ag_news' | 'imdb' | 'yelp_polarity'
N_EVAL = 200          # attacked examples per attack (paper: 1000). 200 is enough to see whether the reproduction works
BASE_K = 30           # sampling size k of the paper's projection (repo default)
GAN_SEARCH_TRIALS = 8   # number of InfoGANs to train; the best by validation reconstruction loss is kept (0 = a single GAN)  # was: 8 to train 8 InfoGANs and keep the best by validation reconstruction loss
ATTACKS = ('pwws', 'textfooler')   # add 'bertattack' last: it is by far the slowest (hours on a T4)
WORK = '/content/drive/MyDrive/tmd_run'   # shared by notebooks 00-04

lab = T.Lab(DATASET, work=WORK)
lab.finetune('base')
E_tr, Y_tr = lab.embeddings('train', 'base')
E_va, Y_va = lab.embeddings('val', 'base')
E_te, Y_te = lab.embeddings('test', 'base')
gan = lab.get_gan('base', E_tr, E_va)
if GAN_SEARCH_TRIALS:   # the authors kept the best of a random hyperparameter search (by validation loss)
    gan, hp = T.search_gans(lab, 'base', E_tr, E_va, trials=GAN_SEARCH_TRIALS)

In [ ]:
# Diagnostic (no attacks, takes seconds): is the generator collapsed to roughly one prototype per cluster?
Et = E_te[:2000].to(T.DEV)
with torch.no_grad():
    outs = [T.project(gan, Et, k=BASE_K)[0] for _ in range(10)]               # 10 independent projections
    preds = torch.stack([lab.model.classifier(o).argmax(-1) for o in outs])
    flip = (preds != preds[0]).any(0).float().mean().item()
    d_proj = torch.norm(outs[0] - Et, dim=-1).mean().item()
    d_draws = torch.norm(outs[0] - outs[1], dim=-1).mean().item()
    d_real = torch.norm(Et - Et[torch.randperm(len(Et), device=T.DEV)], dim=-1).mean().item()
    cte = gan.dis(Et)[1].argmax(-1).cpu()
    ctr = torch.cat([gan.dis(E_tr[i:i + 4096].to(T.DEV))[1].argmax(-1).cpu() for i in range(0, len(E_tr), 4096)])
maj = torch.stack([torch.bincount(Y_tr[ctr == k], minlength=lab.cfg['n_cls']).argmax() if (ctr == k).any()
                   else torch.tensor(0) for k in range(gan.code_dim)])
print(f'prediction changes across 10 projections for {100 * flip:.1f}% of texts')
print(f'||t_hat - t|| = {d_proj:.3f} | between two draws of the same text = {d_draws:.3f} | between random real texts = {d_real:.3f}')
print('clusters used:', len(cte.unique()), 'of', gan.code_dim)
print('accuracy of cluster -> majority label on test:', round((maj[cte] == Y_te[:2000]).float().mean().item() * 100, 1))


In [ ]:
vanilla = T.Victim(lab, mode='none')
base = T.Victim(lab, gan=gan, mode='tmd', proj_mode='sample', k=BASE_K)
res_vanilla = T.evaluate(lab, vanilla, 'vanilla', E_te, Y_te, n=N_EVAL, attacks=ATTACKS)
res_base = T.evaluate(lab, base, 'baseline', E_te, Y_te, n=N_EVAL, attacks=ATTACKS)

In [ ]:
import pandas as pd
paper = T.PAPER[DATASET]
rows = []
for i, m in enumerate(T.METRICS):
    if m not in res_vanilla or m not in res_base: continue   # attacks not run yet
    rows.append({'metric': T.METRIC_LABEL[m],
                 'paper vanilla': paper['vanilla'][i], 'our vanilla': round(res_vanilla[m]['acc'], 1),
                 'paper TMD': paper['TMD'][i], 'our TMD (baseline)': round(res_base[m]['acc'], 1)})
display(pd.DataFrame(rows))
print('Reproduction check: our TMD should be close to the paper TMD column, and clearly above our vanilla.')
print('Per-cell noise at n=1000 is about +-3 points for the attack columns.')
display(T.compare(lab, 'baseline'))